In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [6]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [7]:
# {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
# Best C-index: 0.7380676257590892

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

'''best_model = RandomSurvivalForest(
    bootstrap = True, 
    max_depth = 10, 
    max_features = 'sqrt', 
    min_samples_leaf = 10,
    min_samples_split = 2, 
    n_estimators = 200,
    random_state=12)'''

param_grid = {
    'n_estimators': [100, 200, 500],    
    'max_depth': [5, 10, 15, None],    
    'max_features': ['sqrt', 'log2', 0.5],    
    'min_samples_leaf': [3, 10, 20],
    'n_controls_per_case': [1, 2, 4, 10, 15]
}

In [8]:
# Balanced Random Survival Forest with internal subsampling
class BalancedRandomSurvivalForest:
    """
    Random Survival Forest with balanced case-control sampling per tree.
    Each tree in the forest performs its own case-control subsampling during bootstrap.
    """
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt',
                 min_samples_leaf=1, min_samples_split=2, n_controls_per_case=1,
                 random_state=None, n_jobs=-1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.n_controls_per_case = n_controls_per_case
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.trees = []
        
    def _balanced_bootstrap_sample(self, X, y, random_state):
        """
        Create a balanced bootstrap sample with n_controls_per_case controls for each case.
        """
        rng = np.random.RandomState(random_state)
        
        # Separate cases (events) and controls (censored)
        case_idx = np.where(y['event'] == True)[0]
        control_idx = np.where(y['event'] == False)[0]
        
        # Bootstrap sample cases (with replacement)
        n_cases = len(case_idx)
        sampled_case_idx = rng.choice(case_idx, size=n_cases, replace=True)
        
        # For each case, sample n_controls_per_case controls
        n_controls = n_cases * self.n_controls_per_case
        if len(control_idx) > 0:
            sampled_control_idx = rng.choice(control_idx, size=min(n_controls, len(control_idx)), replace=True)
        else:
            sampled_control_idx = np.array([])
        
        # Combine indices
        sampled_idx = np.concatenate([sampled_case_idx, sampled_control_idx])
        rng.shuffle(sampled_idx)
        
        return X.iloc[sampled_idx], y[sampled_idx]
    
    def fit(self, X, y):
        """
        Fit the forest with balanced sampling per tree.
        """
        self.trees = []
        
        # Set up random states for each tree
        if self.random_state is not None:
            rng = np.random.RandomState(self.random_state)
            tree_seeds = rng.randint(0, 2**31, size=self.n_estimators)
        else:
            tree_seeds = [None] * self.n_estimators
        
        # Train each tree with its own balanced bootstrap sample
        for i in range(self.n_estimators):
            # Create balanced bootstrap sample for this tree
            X_boot, y_boot = self._balanced_bootstrap_sample(X, y, tree_seeds[i])
            
            # Create and train a single tree (RSF with n_estimators=1)
            tree = RandomSurvivalForest(
                n_estimators=1,
                max_depth=self.max_depth,
                max_features=self.max_features,
                min_samples_leaf=self.min_samples_leaf,
                min_samples_split=self.min_samples_split,
                bootstrap=False,  # We already did our own sampling
                random_state=tree_seeds[i],
                n_jobs=1
            )
            
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
        
        return self
    
    def predict(self, X):
        """
        Predict risk scores by averaging across all trees.
        """
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(predictions, axis=0)
    
    def predict_survival_function(self, X):
        """
        Predict survival functions by averaging across all trees.
        """
        # Get survival functions from all trees
        all_surv_funcs = [tree.predict_survival_function(X) for tree in self.trees]
        
        # Average the survival functions
        n_samples = len(X)
        averaged_funcs = []
        
        for i in range(n_samples):
            # Get all functions for this sample across all trees
            sample_funcs = [surv_funcs[i] for surv_funcs in all_surv_funcs]
            
            # Create averaged function
            def make_avg_func(funcs):
                def avg_func(t):
                    return np.mean([f(t) for f in funcs])
                return avg_func
            
            averaged_funcs.append(make_avg_func(sample_funcs))
        
        return averaged_funcs
    
    def score(self, X, y):
        """
        Calculate concordance index.
        """
        predictions = self.predict(X)
        return concordance_index_censored(y['event'], y['time'], predictions)[0]

In [9]:
results_df, best_params, summary = generic_grid_search(
    BalancedRandomSurvivalForest,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw', 'int-brier-score'],
    model_name='BalancedRSF',
    scale_data=False,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1
)


BALANCEDRSF GRID SEARCH (Manual Implementation)
Model: BalancedRandomSurvivalForest
Scaling: No
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - n_estimators: [100, 200, 500]
  - max_depth: [5, 10, 15, None]
  - max_features: ['sqrt', 'log2', 0.5]
  - min_samples_leaf: [3, 10, 20]
  - n_controls_per_case: [1, 2, 4, 10, 15]
Total combinations: 540

[1/540] Testing params: {'n_estimators': 100, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 3}, n_controls_per_case=1
  ⚠️ Error: dataset and feature_columns must be provided when using subsampling
[2/540] Testing params: {'n_estimators': 100, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 3}, n_controls_per_case=2
  Elapsed: 0.0min, Est. remaining: 0.1min
  ⚠️ Error: dataset and feature_columns must be provided when using subsampling
[3/540] Testing params: {'n_estimators': 100, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 3}, n_controls_per_case=4
  Elapse

In [10]:
results_df.to_csv("../results/subsampling/balancedrsf_subsampling_grid_search_results.csv", index=False)

# Train and Test Final Model with Best Hyperparameters

In [11]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/balancedrsf_subsampling_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  n_estimators: {int(best_params['n_estimators'])}")
max_depth_val = best_params['max_depth']
if pd.isna(max_depth_val):
    print(f"  max_depth: None")
else:
    print(f"  max_depth: {int(max_depth_val)}")
print(f"  max_features: {best_params['max_features']}")
print(f"  min_samples_leaf: {int(best_params['min_samples_leaf'])}")
print(f"  n_controls_per_case: {int(best_params['n_controls_per_case'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

FileNotFoundError: [Errno 2] No such file or directory: '../results/corrected_sub_results/balancedrsf_subsampling_grid_search_results.csv'

In [ ]:
# No external subsampling needed - it's done inside each tree!
# We train on the full training set
print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Each tree will perform balanced bootstrap with {int(best_params['n_controls_per_case'])} controls per case")

Original training samples: 2376
Subsampled training samples: 135
Test samples: 594


In [ ]:
# Train final model with best hyperparameters
max_depth_val = best_params['max_depth']
max_depth = None if pd.isna(max_depth_val) else int(max_depth_val)

# Handle max_features - convert to proper type
max_features_val = best_params['max_features']
if isinstance(max_features_val, str):
    # Check if it's a numeric string
    try:
        max_features = float(max_features_val)
    except ValueError:
        # It's a string like 'sqrt' or 'log2'
        max_features = max_features_val
else:
    max_features = max_features_val

final_model = BalancedRandomSurvivalForest(
    n_estimators=int(best_params['n_estimators']),
    max_depth=max_depth,
    max_features=max_features,
    min_samples_leaf=int(best_params['min_samples_leaf']),
    n_controls_per_case=int(best_params['n_controls_per_case']),
    random_state=12,
    n_jobs=-1
)

final_model.fit(x_train, y_train)
print("Model trained successfully with balanced sampling inside each tree!")

Model trained successfully!


In [ ]:
# Test the model on test set
# Calculate c-index
c_index = final_model.score(x_test, y_test)

# Calculate c-index IPCW
tau = y_train["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train, y_test, final_model.predict(x_test), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test)

# Use robust time filtering
max_time = min(y_train['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train, y_test, preds_at_times, times)

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")

\nTest Set Performance:
  C-index: 0.8066
  C-index IPCW: 0.7440
  Integrated Brier Score: 0.0184
